# Regional CROCO hindcast from scratch — a step-by-step guide
### GLORYS + ERA5 hindcast · tides (TPXO) · rivers (GloFAS) · OpenMP · offline nesting
!["hindcast schematic"](figures/croco_hindcast_schematic_GGOSSS2026.png)


### worked example: the Gulf of Guinea `GoG_12` (1/12°) and its nested child `IGOG_36` (1/36°), 1–15 June 2026

**Instructors:** 
- Fernand Assene   — NASMOST, University of Ebolowa, Kribi, Cameroon — fernandassene@hotmail.com
- Babette Tchonang — US Navy Laboratory & Florida State University   — bt25g@fsu.edu 
- Hugo Plombat     — University of Michigan                          — plombat@umich.edu
---

**Notes:** this notebook takes you from a bare machine to a finished CROCO hindcast, then to a finer model **nested** inside
it. Every command is explained before it is run: *what* it does, *why* it is needed, and *what a correct result
looks like*. You can read it as a course, or execute it cell by cell.

**What you write vs. what the system derives**

| you write | the system derives and writes |
|---|---|
| one command: `new_config.sh NAME lon_min lon_max lat_min lat_max [--res N] [--start D --end D] [--tides] [--rivers] [--parent P]` → `domain.cfg` | grid spacing, download boxes, **time step**, data months, number of cycles, **open boundaries from the land mask**, `crocotools_param*.py`, `cppdefs.h`, `param.h`, `jobcomp`, `croco.in`, the binaries |

**The system in one table**

| item | where |
|---|---|
| root folder | `~/croco_work` (`CROCO_ROOT`) |
| Python tools + command line | `gtools/`, `python gtools/ggosss26.py <subcommand>` |
| conda environment | `ggosss26` |
| one region | `hindcast/configs/<NAME>/domain.cfg` → `build_config.sh` → `run_hindcast_cycle.sh` |
| work files / results | `hindcast/scratch/<NAME>/` / `hindcast/model-runs/<NAME>/` |

### How to use this notebook
* Start Jupyter inside the environment, from the repo: `conda activate ggosss26; cd ~/croco_work; jupyter-lab`.
* Shell steps are `%%bash` cells; each sets `H=~/croco_work/hindcast` and `NAME=…` (change `NAME` for another
  region). The scripts load `env.sh`, the conda environment and remove conda for compiling by themselves.
* ⏱ marks long cells (downloads, library builds, the hindcast). You may paste those commands into a terminal
  with `nohup … &` instead.
* Every file and setting is documented in `hindcast/README.md`; the history of changes is in `CHANGES.md`.

---
# Part 0 — the ideas in five minutes
Read this once; every later step refers back to it.

* **Hindcast.** A model run over a *past* period, driven by the best available reconstructions of the ocean and
  the atmosphere. It reconstructs what happened, at a resolution the global products do not have.
* **Downscaling.** Our regional model (1/12°, ~9 km) takes its **initial state** and its **open-boundary values**
  from a global ocean product, and adds the local detail its finer grid and the atmosphere produce.
* **GLORYS12** (Copernicus Marine, `cmems_mod_glo_phy_my_0.083deg_P1D-m`) is the global ocean reanalysis we
  use: daily means of temperature, salinity, currents and sea level, 1/12°, 50 depth levels. It lags the present
  by a few months (in September 2026 it ends on 23 June 2026).
* **ERA5** (ECMWF) is the atmospheric reanalysis: hourly wind, air temperature, humidity, rain, radiation and
  pressure. CROCO computes the air–sea fluxes itself from these (bulk formulae).
* **ini / bry.** The *initial condition* file (the 3-D ocean at the start) and the *boundary* file (the ocean along
  the open edges, in time). Both are interpolated from GLORYS onto our grid.
* **Sigma levels.** CROCO's vertical levels follow the terrain: N = 50 levels between the sea floor and the surface
  in every column. GLORYS uses fixed depths, so the ini/bry tools interpolate between the two.
* **Yorig.** Every date in every file is counted from one origin, **1993-01-01** here. Mixing origins does not crash:
  it silently shifts every date. It is set once, in `hindcast/lib/common.sh`.
* **Spin-up.** A model started from interpolated fields needs a short time to adjust (fast waves, the tide, the
  fine scales). Each hindcast starts with a spin-up run whose final state (a *restart*) starts the real run.
* **Time step and CFL.** The model is stable only if a wave or current crosses less than about one cell per step.
  Finer grid → smaller step: 300 s at 1/12°, 100 s at 1/36°.
* **Online forcing.** The atmosphere is not pre-interpolated: CROCO reads the ERA5 files and interpolates them onto
  its grid while it runs (`ONLINE` + `ERA_ECMWF` in `cppdefs.h`). A finer grid can therefore reuse the same files.
* **Offline nesting.** A finer *child* model inside a coarser *parent*: the parent's output plays the role of the
  global product for the child. Part G does this for `IGOG_36` inside `GoG_12`.

---
# Part A — one-time machine setup
Done **once per machine**. Every step ends with a check cell; skip any step whose check already passes.

## A.1 Linux and build tools
On Windows, CROCO runs in WSL2 Ubuntu (`wsl --install -d Ubuntu` in an Administrator PowerShell). In Ubuntu:
```bash
sudo apt update                                   # refresh the package list
sudo apt install -y build-essential gfortran m4 curl wget git bc \
                    libcurl4-openssl-dev zlib1g-dev nco gettext-base
```
* `build-essential` (gcc, make) and `gfortran`: compile the NetCDF libraries and CROCO itself;
* `m4`: needed by the NetCDF build; `curl`, `wget`, `git`: fetch sources and data;
* `libcurl4-openssl-dev`, `zlib1g-dev`: remote access and compression inside NetCDF/HDF5;
* `bc`: exact arithmetic in the scripts (grid spacing 1/RES); `nco`: `ncrcat` and friends; `gettext-base`: `envsubst`.

**▶ What this cell does.** Prints the kernel and compiler versions, and checks for three small tools the scripts call.

**Why.** CROCO is Fortran: it needs `gfortran`, and the same compiler family must build NetCDF (A.4). `bc` computes the grid spacing 1/RES exactly, `ncdump` inspects NetCDF files, `ncrcat` joins output files.

**✅ What to look for.** gcc and gfortran versions printed, and a path for each of `bc`, `ncrcat`, `ncdump`.

In [ ]:
%%bash
uname -a; gcc --version | head -1; gfortran --version | head -1
which bc ncrcat ncdump || echo "install the missing tools with apt (A.1)"

## A.2 Conda and the `ggosss26` environment
```bash
cd ~ && wget -c https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh   # the conda installer
bash Miniconda3-latest-Linux-x86_64.sh          # accept, default path, "yes" to init; then: source ~/.bashrc
cd ~/croco_work
conda env create -f ggosss2026_conda_env.yaml             # once: creates the env `ggosss26` with every Python package listed
conda activate ggosss26                         # use it in this terminal
```
A separate environment keeps the pre-processing packages (and their versions) apart from the rest of the machine;
`ggosss2026_conda_env.yaml` is the exact list, so another machine gets the same set.

**▶ What this cell does.** Activates the environment and imports every Python package the tools need.

**Why.** The pre-processing (downloads, ini/bry, tides, rivers, nesting) is Python: xarray/netCDF4 read and write the files, `copernicusmarine` downloads GLORYS, `zarr`/`fsspec`/`aiohttp` read ERA5 from the ECMWF ARCO store, `cdsapi` talks to the Copernicus data stores.

**✅ What to look for.** `ggosss26 env OK`.

**⚠️ Classic trap.** `aiohttp` older than 3.9 (an old 0.x copy can be left in an environment) breaks the ERA5 reader: `pip install -U 'aiohttp>=3.9'`.

In [ ]:
%%bash
source ~/miniconda3/etc/profile.d/conda.sh; conda activate ggosss26
python -c "import xarray, copernicusmarine, netCDF4, numpy, scipy, cdsapi, zarr, fsspec, aiohttp; print('ggosss26 env OK, aiohttp', aiohttp.__version__)"

## A.3 `env.sh` — the shared paths
`env.sh` exports `CROCO_ROOT=~/croco_work`, the CROCO source, the data and tools folders, the compilers and their
flags. It also puts `opt_seq/bin` (the NetCDF built in A.4) first on the `PATH`. Every script sources it.

**▶ What this cell does.** Loads `env.sh` and prints three of the paths it defines.

**Why.** Every script finds its files through these variables. If they are wrong, nothing else can work, so it is the first thing to check on a new machine.

**✅ What to look for.** `CROCO_ROOT=/home/<you>/croco_work`, `GTOOLS_DIR=…/croco_work/gtools`, `CROCO_DATA_ROOT=…/croco_work/data`.

In [ ]:
%%bash
source ~/croco_work/env.sh
echo "CROCO_ROOT=$CROCO_ROOT"; echo "GTOOLS_DIR=$GTOOLS_DIR"; echo "CROCO_DATA_ROOT=$CROCO_DATA_ROOT"

## A.4 The NetCDF libraries, built from source into `opt_seq/` ⏱
CROCO reads and writes NetCDF through the Fortran library. Build order **HDF5 → netcdf-c → netcdf-fortran**, all
with the same compilers as CROCO:
```bash
cd ~/croco_work
bash install/00_download_libraries.sh       # fetches the three source archives into install/
bash install/01_build_hdf5.sh               # HDF5: the storage layer under NetCDF-4 (compression, large files)
bash install/02_build_netcdf_c.sh           # NetCDF-C on top of HDF5: the library that reads/writes .nc files
bash install/03_build_netcdf_fortran.sh     # the Fortran interface CROCO calls; ends with ">>> sequential NetCDF stack complete"
```
Each script configures, compiles and installs into `~/croco_work/opt_seq/` ("seq" = sequential: no MPI), so
the system never depends on the NetCDF version of the Linux distribution or of conda.

**▶ What this cell does.** Checks the three libraries exist and asks `nf-config` how to link against them.

**Why.** `jobcomp` (the CROCO build script) calls `nf-config` to get the link flags. It must be *this* build, not the NetCDF inside conda, which was compiled differently and fails to link (`CURL_OPENSSL` errors).

**✅ What to look for.** three `ok`; `nf-config` in `…/croco_work/opt_seq/bin`; flags naming `-lnetcdff -lnetcdf`.

**⚠️ Classic trap.** The stack cannot be moved: relocating `~/croco_work` means rebuilding it.

In [ ]:
%%bash
source ~/croco_work/env.sh >/dev/null
for f in libhdf5.so libnetcdf.so libnetcdff.so; do [ -f $CROCO_ROOT/opt_seq/lib/$f ] && echo "ok      $f" || echo "MISSING $f"; done
which nf-config; nf-config --prefix; nf-config --flibs

## A.5 CROCO v2.1.3 and the pre-processing tools ⏱
```bash
source ~/croco_work/env.sh; conda activate ggosss26
bash ~/croco_work/install/04_get_croco.sh      # -> code/croco, code/croco_pytools
```
Two sets of Python pre-processing tools are used, for two jobs:
* `code/croco_pytools` (v2.0.4) builds the **grid**.
* `gtools/croco_pytools` (shipped in this package) builds the **ini/bry and tide** files. It carries fixes for
  domains that cross the 0° meridian, like the Gulf of Guinea.

**▶ What this cell does.** Lists a few key files of the CROCO source and of both tool sets.

**Why.** A missing file here is the usual cause of a failure much later (e.g. `ModuleNotFoundError: ibc_class` during step 04).

**✅ What to look for.** every path listed, no `No such file`.

In [ ]:
%%bash
source ~/croco_work/env.sh >/dev/null
ls $CROCO_MODEL_DIR/OCEAN/cppdefs.h $CROCO_PYTOOLS_DIR/prepro/Modules/toolsf*.so
ls $GTOOLS_DIR/croco_pytools/prepro/Modules/ibc_class.py $GTOOLS_DIR/nesting.py $GTOOLS_DIR/download/ERA5/era5_arco.py

## A.6 Static datasets: topography, coastline, tides, river mouths ⏱
```bash
mkdir -p ~/croco_work/data && cd ~/croco_work/data
wget -c https://data-croco.ifremer.fr/DATASETS/DATASETS_CROCOTOOLS.tar.gz && tar -xzf DATASETS_CROCOTOOLS.tar.gz
```

**▶ What this cell does.** Lists the four static files the system reads.

**Why.** ETOPO2 gives the bathymetry and GSHHS the coastline of the grid (step 01); TPXO7 gives the tidal harmonics (step 09); the Dai & Trenberth file gives the positions and names of river mouths (step 10).

**✅ What to look for.** four files listed.

In [ ]:
%%bash
D=~/croco_work/data/DATASETS_CROCOTOOLS
ls -l $D/Topo/etopo2.nc $D/gshhs/GSHHS_shp/i/GSHHS_i_L1.shp $D/TPXO7/TPXO7.nc $D/RUNOFF_DAI/Dai_Trenberth_runoff_global_clim.nc

## A.7 Accounts (once)
* **Copernicus Marine** (GLORYS): register at <https://data.marine.copernicus.eu/register>, then
  `conda activate ggosss26; copernicusmarine login`.
* **Copernicus Climate Data Store** (ERA5): at <https://cds.climate.copernicus.eu>, copy the *Personal Access Token*
  from your profile, then `printf "url: https://cds.climate.copernicus.eu/api\nkey: <token>\n" > ~/.cdsapirc;
  chmod 600 ~/.cdsapirc`, and accept the licence of *ERA5 hourly data on single levels*. The same token opens the
  ERA5 ARCO store (B.3).
* **Early Warning Data Store** (GloFAS, only for rivers): same ECMWF account, other store:
  `printf "url: https://ewds.climate.copernicus.eu/api\nkey: <token>\n" > ~/.ewdsapirc; chmod 600 ~/.ewdsapirc`,
  and accept the licence at <https://ewds.climate.copernicus.eu/datasets/cems-glofas-historical?tab=download>.

Keep these files private (`chmod 600`) and never paste a token into a notebook or a message.

**▶ What this cell does.** Checks the three accounts without printing any secret.

**Why.** A download step that fails on authentication after an hour of waiting is frustrating: checking once at the start avoids it.

**✅ What to look for.** `CMEMS login OK`, `CDS client OK`, and `EWDS configured` if you want rivers.

In [ ]:
%%bash
source ~/miniconda3/etc/profile.d/conda.sh; conda activate ggosss26
copernicusmarine login --check-credentials-valid >/dev/null 2>&1 && echo "CMEMS login OK" || echo "CMEMS: run copernicusmarine login"
python -c "import cdsapi; cdsapi.Client(); print('CDS client OK')"
[ -f ~/.ewdsapirc ] && echo "EWDS configured (rivers possible)" || echo "no ~/.ewdsapirc: fine unless RIVERS=1"

---
# Part B — build the parent configuration `GoG_12`

## B.0 Describe the region: one command
A configuration is described by **one file**, `hindcast/configs/<NAME>/domain.cfg`, written by `new_config.sh`.
GoG_12 was created with:
```bash
~/croco_work/hindcast/new_config.sh GoG_12 -10 12.5 -6 8 --res 12 --start 2026-06-01 --end 2026-06-17 \
        --tides --rivers --threads 8 --hcast-days 16
```
* box 10°W–12.5°E, 6°S–8°N at 1/12°;
* period **1–16 June 2026** (the end date is exclusive: the last model time is 17 June 00:00). We analyse
  1–15 June; the extra day only gives the nested child (Part G) boundary data for its last day;
* `--hcast-days 16` makes the whole period **one continuous run** (1 cycle), preceded by a 2-day spin-up;
* tides and rivers on, 8 OpenMP threads.

**▶ What this cell does.** Shows the non-comment lines of `domain.cfg`, then everything the system derives from it.

**Why.** Every later step reads these derived values, so this summary is the place to catch a wrong box, period or option before anything is downloaded or compiled.

**✅ What to look for.** cpp key `GOG_12`; **dt 300 s**; period 2026-06-01 → 06-17 = **1 cycle** of 2 d spin-up + 16 d; ocean `glorys (GLORYS)`, box **−11.50, 14.00, −7.50, 9.50** (the grid + 1.5°), months **2026-05 … 2026-06**; ERA5 months 2026-05 … 2026-06; tides=1 rivers=1 → build `plain_tides_rivers`, threads 8. The boundaries show `?` until step 02 has read the land mask.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
grep -vE "^\s*#|^\s*$" $H/configs/$NAME/domain.cfg
bash -c "source $H/lib/common.sh $NAME >/dev/null && show_config"

### The fast way, and the step-by-step way
```bash
~/croco_work/hindcast/build_config.sh GoG_12            # steps 01-10 in order, logs in scratch/GoG_12/build_logs/
~/croco_work/hindcast/build_config.sh GoG_12 --from 4   # continue from a step
~/croco_work/hindcast/build_config.sh GoG_12 --only 2   # one step
```
The sections below run the **same steps one at a time**, so that you see what each one produces.

### Every script, what it reads, what it writes
| script | reads | writes | role |
|---|---|---|---|
| `new_config.sh` | your arguments (and the parent's `domain.cfg` with `--parent`) | `configs/<NAME>/domain.cfg` | describes a region: the only hand-edited file |
| `lib/common.sh` | `domain.cfg`, `obc.cfg` | nothing (exports variables) | loaded by every script: derives dt, boxes, months, cycles, folders, the build name |
| `build_config.sh` | `domain.cfg` | logs in `scratch/<NAME>/build_logs/` | runs steps 01–10 in order; `--from N`, `--only N`, `--no-test` |
| `steps/01_make_grid.sh` | ETOPO2, GSHHS | `CROCO_FILES/croco_grd.nc`, `configs/<NAME>/grid.ini` | the grid, land mask, smoothed bathymetry |
| `steps/02_check_grid.sh` | the grid | `obc.cfg`, `crocotools_param.py`, `crocotools_param_tides.py`, `CARD.md` | open boundaries from the mask; CFL check |
| `steps/03_download_data.sh` | accounts (A.7) | `downloaded_data/GLORYS/YYYY_MM.nc`, `downloaded_data/ERA5/for_croco/*.nc` | ocean + atmosphere data (a nested child: converts the parent instead) |
| `steps/04_make_ini_bry.sh` | GLORYS files, the grid | `croco_ini_*.nc`, `croco_bry_*.nc` | initial and boundary files of the proof run |
| `steps/05_patch_source_files.sh` | CROCO templates, grid, `obc.cfg` | `configs/<NAME>/cppdefs.h`, `param.h`, `jobcomp` | compile-time settings |
| `steps/06_patch_croco_in.sh` | derived values | `configs/<NAME>/croco.in` | run-time settings |
| `steps/07_compile.sh` | the three source files | `scratch/<NAME>/croco_<build>` | one binary per physics combination, OpenMP if `NTHREADS > 1` |
| `steps/08_run_test.sh` | the binary, step-04 files, ERA5 | `scratch/<NAME>/test_<build>/` | 7-day proof run |
| `steps/09_tides.sh` | TPXO7 | `croco_frc.nc` | tide forcing for the proof run |
| `steps/10_rivers_glofas.sh` | GloFAS (EWDS), Dai mouths | `croco_runoff.nc`, `for_croco_in.txt` | river discharge and positions |
| `run_hindcast_cycle.sh` | everything above | `model-runs/<NAME>/<date>_<build>/` | the hindcast: spin-up + hindcast per cycle, its own tide/ini/bry files |
| `notebooks/verify_config.py` | the run outputs | a report | continuity and plausibility checks |
| `gtools/ggosss26.py` | — | — | the Python command line all steps call (downloads, ini/bry, tides, rivers) |
| `gtools/nesting.py` | a parent's `croco_avg.nc` | GLORYS-format monthly files | offline nesting converter (Part G) |

## B.1 Step 01 — the grid

**▶ What this cell does.** Writes `configs/GoG_12/grid.ini` (box, resolution) and runs the grid builder: horizontal coordinates, ETOPO2 bathymetry, GSHHS land mask, then smoothing of the bathymetry.

**Why.** Everything else lives on this grid. The smoothing matters: steep slopes on sigma levels create false currents (the *pressure-gradient error*); the builder smooths until the slope factor `rx0` is at most 0.2.

**✅ What to look for.** `xi_rho = 273`, `eta_rho = 171` in `scratch/GoG_12/CROCO_FILES/croco_grd.nc`. The model's interior is 2 points smaller (271 × 169): step 05 writes that into `param.h` automatically.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/01_make_grid.sh $NAME

## B.2 Step 02 — open boundaries, parameter files, stability check

**▶ What this cell does.** Reads the land mask along each edge and declares an edge **open** if more than half of it is ocean. It writes `obc.cfg`, `crocotools_param.py` (ini/bry settings), `crocotools_param_tides.py` (tide settings) and `CARD.md` (a one-page summary of the region), and prints the Courant numbers for the derived time step.

**Why.** An open boundary must be declared identically in three places (the ini/bry tools, `cppdefs.h`, the boundary file). Deriving all three from the mask makes them agree by construction. The Courant numbers tell you, before compiling, whether dt is stable.

**✅ What to look for.** south and west **open** (mostly ocean), east and north **closed** (the Gabon/Cameroon and West-African coasts); `rx0 = 0.200`; barotropic Courant **0.13** (well below ~0.7).

**⚠️ Classic trap.** A borderline edge (40–60 % ocean) may deserve a manual decision: `OBC="S W E N"` (1 open / 0 closed) in `domain.cfg`, then re-run from step 02.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/02_check_grid.sh $NAME
cat $H/configs/$NAME/CARD.md

## B.3 Step 03 — the ocean (GLORYS) and the atmosphere (ERA5) ⏱
**GLORYS.** Months 2026-05 and 2026-06 (the spin-up starts on 30 May; boundaries need a day on each side).
The downloader asks for the **whole period in one request** from Copernicus Marine's *time-series* store, then splits
it into monthly files `2026_05.nc`, `2026_06.nc`. Why this matters: the store is cut into chunks; the *map* store's
chunks are one day × one level × 512×2048 points, so a small regional box downloads 17–60× more data than it keeps,
while the time-series store's chunks fit a long regional request much better. For GLORYS these chunks are very long
in time (2081 days × one level × 16×16 points), so the server still sends far more than we keep, and the download
takes hours even on a saturated link. Two safeguards make it reliable:
* **pieces**: the request is split into 21 pieces, one variable × one depth band (0–50, 50–250, 250–1100,
  1100–3400, 3400–6000 m), plus sea level. Each chunk holds one level of one variable, so this costs no extra data,
  but a network failure repeats one piece (minutes) instead of the whole request (hours), and a re-run skips the
  pieces already on disk;
* **watchdog**: after a network drop the download library can wait forever on a dead connection. Each piece runs
  in a child process, and if it receives no data for 10 minutes it is restarted (up to 5 times).

**ERA5.** Read directly from ECMWF's **ARCO store** (analysis-ready Zarr, the same data as the Climate Data Store,
without the request queue), one read per variable for the whole period. Three fields CROCO needs are not stored
as such and are computed:
* `Q`, specific humidity, from the 2 m dew point and the sea-level pressure (Bolton 1980);
* `SSR`, the **net** solar flux (what CROCO absorbs), from the downward solar flux `ssrd` and its direct part
  `fdir`, with the open-water albedo of Taylor et al. (1996): `α_dir = 0.037 / (1.1 μ^1.4 + 0.15)` for the direct
  beam (μ = cosine of the sun's zenith angle), 0.06 for diffuse light;
* `LSM`, the land mask, where the ERA5 sea temperature is missing.

The files are written as CROCO reads them: `<VAR>_Y2026M06.nc` (T2M, Q, TP, SSR, STRD, U10M, V10M, msl, SST, LSM),
time in **days since 1993-01-01**, latitude increasing, missing value 9999. Accumulated fields are converted to
rates: precipitation ×1000/3600 → kg m⁻² s⁻¹, radiation /3600 → W m⁻².

**▶ What this cell does.** Downloads GLORYS for the derived months and ERA5 for the period, then lists the files with their number of time records and depth levels.

**Why.** These two products drive everything: GLORYS gives the ocean state and the open-boundary values, ERA5 the surface fluxes. Re-running is safe: finished months are skipped.

**✅ What to look for.** `2026_05.nc` and `2026_06.nc` with `depth = 50` and `time` = 31 and 23 (GLORYS ends on 23 June); 20 ERA5 files (10 × 2 months) in `for_croco/`.

**⚠️ Classic trap.** A monthly-mean GLORYS product (one record per month) cannot build ini/bry: the daily `P1D-m` product is required.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
# in a terminal you may prefer:  nohup bash $H/steps/03_download_data.sh $NAME > ~/${NAME}_download.log 2>&1 &
bash $H/steps/03_download_data.sh $NAME 2>&1 | grep -v "^INFO" | tail -30

**▶ What this cell does.** Opens one ERA5 file of each kind and prints its time span, units and value range.

**Why.** A wrong unit or time origin does not crash CROCO: it gives a wrong ocean. Checking the numbers once is the cheapest insurance.

**✅ What to look for.** time in `days since 1993-1-1`, 720 hourly records for June; T2M ≈ 287–312 K; Q ≈ 0.009–0.024 kg/kg; SSR 0 at night and up to ~950 W m⁻² at noon; msl ≈ 1005–1021 hPa.

In [ ]:
import os, numpy as np, xarray as xr, datetime as dt
F = os.path.expanduser("~/croco_work/hindcast/scratch/GoG_12/downloaded_data/ERA5/for_croco")
for tag in ["T2M", "Q", "TP", "SSR", "STRD", "U10M", "V10M", "msl"]:
    d = xr.open_dataset(f"{F}/{tag}_Y2026M06.nc", decode_times=False)
    t = d.time.values; v = d[tag].values
    t0 = dt.datetime(1993, 1, 1) + dt.timedelta(days=float(t[0])); t1 = dt.datetime(1993, 1, 1) + dt.timedelta(days=float(t[-1]))
    print(f"{tag:5s} {d[tag].units:11s} {len(t)} records {t0:%m-%d %H:%M} -> {t1:%m-%d %H:%M}   min {np.nanmin(v):10.4g}  max {np.nanmax(v):10.4g}")

## B.4 Step 04 — initial and boundary files for a short proof run

**▶ What this cell does.** Interpolates GLORYS onto the grid: a 3-D initial state on 2 June, and boundary values for the open edges from 1 to 10 June.

**Why.** Before a long run, a 7-day *proof run* (step 08) shows that the whole chain works. The initial date is day 2 of the period so GLORYS has records on both sides; the boundary window is one day longer on each side, because CROCO needs a record before and after every time step.

**✅ What to look for.** `croco_ini_GLORYS_Y2026M06D02.nc` with `s_rho = 50`; a `croco_bry_GLORYS_…nc` with variables on the south and west edges only.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/04_make_ini_bry.sh $NAME

## B.5 Step 05 — the source settings `cppdefs.h`, `param.h`, `jobcomp`
CROCO is configured at **compile time**:
* `cppdefs.h` switches features on and off (`# define` / `# undef`): our configuration key `GOG_12`, its open
  boundaries, the ERA5 online forcing (`ONLINE`, `ERA_ECMWF`), and later tides and rivers;
* `param.h` fixes the array sizes: `LLm0 = 271, MMm0 = 169, N = 50`, read from the grid;
* `jobcomp` is the build script; it needs the path to the CROCO source.

**▶ What this cell does.** Copies the three templates into `configs/GoG_12/` and applies every edit from `domain.cfg`, `obc.cfg` and the grid.

**Why.** These edits are where hand-built configurations usually go wrong (a grid size typed wrong, an open boundary declared in the wrong block). Deriving them removes the typing.

**✅ What to look for.** `# define GOG_12`, the `OBC_*` block matching step 02, `# elif defined GOG_12` with `LLm0=271, MMm0=169, N=50` in `param.h`.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/05_patch_source_files.sh $NAME

## B.6 Step 06 — the run settings `croco.in`
`croco.in` is read when the model starts: number of steps and dt, vertical-grid parameters (must match `param.h`),
input files, output frequency, the sponge and the ERA5 block. For GoG_12: `time_stepping` = 2016 steps of 300 s
(7 days) for the proof run, `S-coord` = `7.0 2.0 200.0`, `online` = `2026 6 24 2026 6` + the ERA5 folder (real
dates, 24 records per day).

**▶ What this cell does.** Writes the whole `croco.in` of the proof run from the derived values.

**Why.** Every number in it must agree with another file (grid, `param.h`, ini/bry dates, ERA5 months); writing it from the same source guarantees they do.

**✅ What to look for.** `>>> croco.in written` and the proof-run dates.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/06_patch_croco_in.sh $NAME

## B.7 Step 07 — compile, and OpenMP
Each combination of optional physics is its own binary: `croco_plain`, `croco_plain_tides`, `croco_plain_rivers`,
`croco_plain_tides_rivers`. With `NTHREADS > 1` the build adds **OpenMP**: the domain is cut into tiles
(`NSUB_X × NSUB_E`) that the cores compute in parallel.

**▶ What this cell does.** Builds `croco_plain` from a clean `Compile/` folder, with conda removed from the environment.

**Why.** The plain model is always built and tested first: if it fails, tides or rivers are not the cause.

**✅ What to look for.** `build plain: # undef TIDES | # undef PSOURCE …`, the CROCO logo, `CROCO is OK`, a file `scratch/GoG_12/croco_plain`.

**⚠️ Classic trap.** Compiling inside conda links conda's NetCDF (`CURL_OPENSSL` errors); the script removes conda itself.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/07_compile.sh $NAME plain

## B.8 Step 08 — the 7-day proof run ⏱ (minutes)

**▶ What this cell does.** Runs the plain model for 7 days from the step-04 files and keeps its outputs in `scratch/GoG_12/test_plain/`.

**Why.** It proves, cheaply, that the initial state, the boundaries, ERA5 and the time origin all agree. The first `time[DAYS]` printed by CROCO is the number of days since 1993-01-01, so it checks Yorig by itself.

**✅ What to look for.** `GET_INITIAL`, `GET_BRY`, `ONLINE_BULK … Read file`, bounded energy, `MAIN: DONE`; first `time[DAYS]` ≈ **12205** (1993-01-01 → 2026-06-02).

**⚠️ Classic trap.** `ERROR in get_bry: cannot read variable 'bry_time'`: the boundary window is too short (a missing month). `Abnormal termination: netCDF INPUT` after *Open Meteo file*: an ERA5 file or month is missing.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/08_run_test.sh $NAME plain

---
# Part C — tides (TPXO)
Global reanalyses contain no tide, but on the Gulf of Guinea shelves the M2 tide is ~0.5 m, and tides drive coastal
sea level, currents and mixing. With `TIDES=1`:

| | without tides | with tides |
|---|---|---|
| data | — | TPXO7, 10 constituents → `croco_frc.nc` |
| `cppdefs.h` | `# undef TIDES` | `# define TIDES` (+ sea-level, current and potential tides, and a ramp) |
| output | 6-hourly history | hourly history (to resolve the tide), daily averages |
| binary | `croco_plain` | `croco_plain_tides` |

The tide file depends on the **start date** (phases are counted from Yorig, and the slow *nodal* modulation of each
constituent is computed for the date), so every run gets its own file.

**▶ What this cell does.** Builds the tide file for the proof run and checks it: constituents, amplitude, and fill values.

**Why.** A tide file with missing values at some points would inject garbage at the boundary. GoG_12 crosses 0°, which used to expose bugs in the tide reader; the check must report **0 fill values**.

**✅ What to look for.** 10 waves, M2 amplitude around half a metre on average, the largest on the shelves, **0** fill values.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/09_tides.sh $NAME

**▶ What this cell does.** Compiles the tide binary and repeats the proof run with it.

**Why.** It isolates the effect of one change: if the plain run worked and this one fails, the tide is the cause.

**✅ What to look for.** `MAIN: DONE`. On the shelf the sea level oscillates with a period close to 12.4 h (M2).

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/07_compile.sh $NAME plain_tides
bash $H/steps/08_run_test.sh $NAME plain_tides

---
# Part D — rivers with GloFAS discharge
Rivers bring fresh water that forms buoyant coastal plumes (the Niger, Volta, Sanaga, Ogooué…). Instead of a
climatology, the hindcast uses **GloFAS**: the daily river discharge of the actual dates.

How the rivers are chosen and placed:
1. the positions and names of river **mouths** come from the Dai & Trenberth file (its discharge is not used);
2. for each mouth inside the grid, the GloFAS cell with the largest mean discharge within 0.25° is its outlet;
3. rivers below 100 m³/s are dropped, and two mouths of the same channel are counted once;
4. each river is moved to the nearest **wet** model cell, and `croco_runoff.nc` gets its daily discharge;
5. `croco.in` gets one line per river with temperature **27 °C** and salinity **0** (`RIVER_TEMP`, `RIVER_SALT`).

**▶ What this cell does.** Downloads GloFAS for the period (only the days the store offers), selects the rivers, places them and writes `croco_runoff.nc` and the `croco.in` river block.

**Why.** A river on a land cell would be silently lost, and a wrong salinity would inject brackish water; the step prints both so you can see them.

**✅ What to look for.** a table of the selected rivers with their mean discharge, *N river(s) positioned on wet cells*, and a `psource_ncfile` count equal to its number of rows.

**⚠️ Classic trap.** The Congo mouth sits on GoG_12's southern edge; its plume comes in through the open boundary with GLORYS. Adding it as a point source next to an open boundary (~40 000 m³/s in one cell) is a stability risk.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/10_rivers_glofas.sh $NAME

**▶ What this cell does.** Compiles the full binary (tides + rivers) and runs the proof run with it.

**Why.** This is the binary of the real hindcast; running it once briefly avoids discovering a problem after hours.

**✅ What to look for.** `GET_PSOURCE -- Read Run-off flow fields`, `MAIN: DONE`, lower surface salinity near the big mouths.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
bash $H/steps/07_compile.sh $NAME plain_tides_rivers
bash $H/steps/08_run_test.sh $NAME plain_tides_rivers

**▶ What this cell does.** Maps the surface salinity with rivers, and its difference with the tide-only run.

**Why.** The difference isolates the river plumes: it shows that the rivers are really in the model, where they are, and how far the fresh water spreads in a week.

**✅ What to look for.** blue patches (fresher water, negative difference) at the river mouths, strongest off the Niger delta.

In [ ]:
import os, numpy as np, xarray as xr, matplotlib.pyplot as plt
B = os.path.expanduser("~/croco_work/hindcast/scratch/GoG_12/")
a = xr.open_dataset(B + "test_plain_tides/croco_his.nc", decode_times=False)
b = xr.open_dataset(B + "test_plain_tides_rivers/croco_his.nc", decode_times=False)
sa = a.salt.isel(time=-1, s_rho=-1).where(a.mask_rho == 1); sb = b.salt.isel(time=-1, s_rho=-1).where(b.mask_rho == 1)
d = sb - sa
print("no rivers: min S %.2f | rivers: min S %.2f | largest freshening %.2f PSU" % (float(sa.min()), float(sb.min()), float(d.min())))
fig, ax = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
h = ax[0].pcolormesh(a.lon_rho, a.lat_rho, sb, cmap="viridis", vmin=30, vmax=36); ax[0].set_title("surface salinity, tides + rivers")
fig.colorbar(h, ax=ax[0], extend="min")
h = ax[1].pcolormesh(a.lon_rho, a.lat_rho, d, cmap="BrBG", vmin=-1, vmax=1); ax[1].set_title("rivers - no rivers (PSU)")
fig.colorbar(h, ax=ax[1], extend="both")
for x in ax: x.set_aspect("equal")
plt.show()

---
# Part E — OpenMP: is the parallel model identical and faster?
With 8 threads the domain is split into tiles computed at the same time. Before trusting it, compare one model day
run sequentially and with OpenMP, from the same inputs.

**▶ What this cell does.** Builds a sequential binary, runs one day with it and with the OpenMP binary, prints both run times and the largest difference of each field after one day.

**Why.** Parallel code can hide bugs that only show as differences (a tile edge computed twice or not at all). The differences must be at the level of round-off, and the speed-up tells you what to expect for the hindcast.

**✅ What to look for.** both runs `MAIN: DONE`; differences tiny compared with the values (round-off level); the OpenMP run several times faster.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
W=~/croco_work/hindcast/scratch/$NAME
bash $H/steps/07_compile.sh $NAME plain 1 && cp $W/croco_plain $W/croco_plain_seq     # 1 thread = sequential
bash $H/steps/07_compile.sh $NAME plain                                              # NTHREADS from domain.cfg
for v in seq omp; do
  R=$W/omp_check/$v; rm -rf $R; mkdir -p $R/CROCO_FILES
  for f in $W/CROCO_FILES/croco_grd.nc $W/CROCO_FILES/croco_ini_*.nc $W/CROCO_FILES/croco_bry_*.nc; do ln -sf $f $R/CROCO_FILES/; done
  cp $H/configs/$NAME/croco.in $R/croco.in
  sed -i "/^time_stepping:/{n; s/.*/                288      300      60      1/}" $R/croco.in     # 1 day
  ( cd $R && source ~/croco_work/env.sh >/dev/null; B=$W/croco_plain; [ $v = seq ] && B=$W/croco_plain_seq
    export OMP_NUM_THREADS=8 OMP_STACKSIZE=512M; ulimit -s unlimited
    t0=$(date +%s); $B croco.in > run.log 2>&1; echo "  $v: $(grep -c 'MAIN: DONE' run.log) MAIN: DONE, $(( $(date +%s)-t0 )) s for 1 model day" )
done

**▶ What this cell does.** Compares the two one-day runs field by field: the largest difference between the sequential and the OpenMP result, next to the largest value of the field.

**Why.** Equal run lengths prove nothing about correctness; the fields themselves must agree. Tiny differences are expected (the order of floating-point sums changes with the tiling), large ones mean a bug.

**✅ What to look for.** every difference many orders of magnitude below the field's size (round-off level).

In [ ]:
import os, numpy as np, xarray as xr
W = os.path.expanduser("~/croco_work/hindcast/scratch/GoG_12/omp_check/")
a = xr.open_dataset(W + "seq/CROCO_FILES/croco_his.nc", decode_times=False)
b = xr.open_dataset(W + "omp/CROCO_FILES/croco_his.nc", decode_times=False)
for v in ["zeta", "temp", "salt", "u", "v"]:
    print(f"{v:5s} max|seq - omp| = {np.nanmax(np.abs(a[v].values[-1] - b[v].values[-1])):.2e}   (max|value| {np.nanmax(np.abs(a[v].values[-1])):.2e})")

---
# Part F — the GoG_12 hindcast, 1–16 June 2026 ⏱
```bash
nohup ~/croco_work/hindcast/run_hindcast_cycle.sh GoG_12 > ~/GoG_12_hindcast.log 2>&1 &
tail -f ~/GoG_12_hindcast.log
```
What the driver does for each cycle (here one cycle, T = 1 June):
0. builds the tide file at the spin-up start (30 May);
1. spin-up initial state from GLORYS on 30 May, boundaries 29 May → 2 June;
2. **spin-up run**, 2 days → a restart file (the ocean state on 1 June 00:00);
3. hindcast boundaries 31 May → 18 June; the initial state is the spin-up restart;
4. **hindcast run**, 16 days → hourly history `croco_his.nc` and daily means `croco_avg.nc`.

```
hindcast/model-runs/GoG_12/20260601_plain_tides_rivers/
├── spinup/  croco.in  croco_spinup.out  CROCO_FILES/{croco_ini, croco_bry, croco_rst, croco_frc, croco_runoff, croco_avg}.nc
├── hcast/   croco.in  croco_hcast.out   CROCO_FILES/{croco_ini (= spin-up restart), croco_bry, croco_his, croco_avg, …}.nc
└── gen_spinup/  gen_hcast/  gen_tides/   the generated input files and their logs
```

**▶ What this cell does.** Shows which binaries exist; the launch line is commented so the notebook does not start a long run by accident.

**Why.** The driver picks the binary from `TIDES`/`RIVERS` in `domain.cfg`; checking it exists first avoids a failure after the downloads.

**✅ What to look for.** `croco_plain_tides_rivers` listed. Uncomment the `nohup` line to launch.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=GoG_12
ls -l ~/croco_work/hindcast/scratch/$NAME/croco_plain* 2>/dev/null
# nohup $H/run_hindcast_cycle.sh $NAME > ~/${NAME}_hindcast.log 2>&1 &     # <- uncomment to launch

**▶ What this cell does.** Checks the run: every cycle finished, the time axis is continuous, the values are physical.

**Why.** A run can end with `MAIN: DONE` and still be wrong (a gap between cycles, a field drifting away). These checks read the outputs themselves.

**✅ What to look for.** the cycle `OK`, SST and salinity ranges plausible for the Gulf of Guinea (surface ~20–30 °C), and the model's mean SST close to GLORYS's.

In [ ]:
%%bash
python3 ~/croco_work/notebooks/verify_config.py GoG_12 plain_tides_rivers

**▶ What this cell does.** Maps sea-surface temperature and surface currents on the last hindcast day.

**Why.** A first look catches what numbers miss: noise at a boundary, a front in the wrong place, a river plume going the wrong way.

**✅ What to look for.** the warm Gulf of Guinea, cooler water in the south (the seasonal upwelling season starts in June), the Guinea Current flowing east along the northern coast.

In [ ]:
import os, glob, numpy as np, xarray as xr, matplotlib.pyplot as plt
HIS = sorted(glob.glob(os.path.expanduser("~/croco_work/hindcast/model-runs/GoG_12/20260601*/hcast/CROCO_FILES/croco_his.nc")))[-1]
ds = xr.open_dataset(HIS, decode_times=False)
t = np.datetime64("1993-01-01") + (ds.scrum_time.values * 1e9).astype("timedelta64[ns]")
k = int(np.argmin(np.abs(t - np.datetime64("2026-06-15T12:00"))))          # a time inside the analysis period
sst = ds.temp.isel(time=k, s_rho=-1).where(ds.mask_rho == 1)
u = ds.u.isel(time=k, s_rho=-1).values; v = ds.v.isel(time=k, s_rho=-1).values
ur = 0.5 * (u[1:-1, :-1] + u[1:-1, 1:]); vr = 0.5 * (v[:-1, 1:-1] + v[1:, 1:-1])
lon, lat = ds.lon_rho.values, ds.lat_rho.values
fig, ax = plt.subplots(figsize=(11, 6))
pc = ax.pcolormesh(lon, lat, sst, shading="auto", cmap="RdYlBu_r"); s = 6
ax.quiver(lon[1:-1:s, 1:-1:s], lat[1:-1:s, 1:-1:s], ur[::s, ::s], vr[::s, ::s], scale=15)
ax.set_title(f"GoG_12 SST + surface currents, {str(t[k])[:16]}"); fig.colorbar(pc, label="°C"); ax.set_aspect("equal"); plt.show()

---
# Part G — offline nesting: `IGOG_36` (1/36°) inside `GoG_12` (1/12°)

## G.0 The idea
GoG_12 downscales GLORYS. Nesting is **the same idea one level down**: a finer model downscales *our own* GoG_12.
```
GLORYS (global, 1/12°)  →  GoG_12 (1/12°, regional)  →  IGOG_36 (1/36°, inner Gulf of Guinea)
     parent of GoG_12          parent of IGOG_36               the child
```
A nested child is built exactly like any configuration (grid, boundaries, ini/bry, compile, run). **Only the source
of the ocean changes**: instead of GLORYS, the child's initial state and boundaries come from the GoG_12 output.
It is *offline* because the parent runs first and the child reads its saved output afterwards (the two models do
not talk to each other while running; that would be *online* nesting, e.g. AGRIF).

### The one new tool: the converter
The ini/bry tools expect GLORYS-like files: fixed depth levels and the names `thetao, so, uo, vo, zos`. CROCO writes
sigma levels and the names `temp, salt, u, v, zeta`. `gtools/nesting.py` translates the parent's output:
* interpolates temperature, salinity and currents from the parent's 50 sigma levels onto the 50 standard GLORYS depths;
* rotates the currents to east/north;
* renames everything, and writes monthly `YYYY_MM.nc` files — the same layout as the GLORYS folder.

After this, the standard chain builds the child's ini/bry **without knowing** that the "global product" is our own model.

### Choices made for the child (all set automatically by `new_config.sh … --parent GoG_12`)
| setting | value | why |
|---|---|---|
| box | 4–12.2°E, 5.5°S–5.5°N | at least 0.3° inside the parent on every side (checked): the ini/bry reader needs the parent's data to cover the child grid + 0.2°, even along land edges. Its open southern edge lies 6 parent cells (~55 km) inside GoG_12's open boundary at 6°S: there the parent's solution is mostly its own GLORYS boundary data, so the child must take its values from further inside |
| resolution, dt | 1/36° (~3 km), **100 s** | three times finer cells → three times shorter step (CFL) |
| parent output used | **daily means** (`croco_avg.nc`) | averaging removes the parent's tide; the child makes its own tide from TPXO, so hourly snapshots taken once a day would alias the tide |
| spin-up | 1 day (31 May) | the interpolated state has no fine-scale structure; the child grows its own within about a day |
| period | 1–15 June (ends one day before the parent) | the parent's daily means are stamped at noon: the last one (16 June 12:00) must come *after* the child's last step (16 June 00:00) |
| sponge | 20 km, 400 m² s⁻¹ | the parent's sharp structures arrive at the child's open edges; the sponge absorbs them instead of reflecting them inward |
| atmosphere | the parent's ERA5 files | CROCO interpolates ERA5 onto whatever grid runs, so the finer grid reuses the same files (nesting refines the ocean, not the atmosphere) |
| tides, rivers | on, with the child's own tide file and river positions | computed on the child grid by steps 09 and 10 |

## G.1 Create the child configuration

**▶ What this cell does.** Creates `configs/IGOG_36/domain.cfg` as a child of GoG_12, then prints what the system derives from it.

**Why.** One command replaces the manual work: `PARENT=GoG_12` switches the ocean source to the parent's output, the period, physics and threads are taken from the parent, the spin-up and sponge are set for a child, and a box outside the parent is refused.

**✅ What to look for.** `NESTED in GoG_12`; period 2026-06-01 → 06-16 = 1 cycle of 1 d spin-up + 15 d; dt **100 s**; `SPONGE="20000. 400."`; ocean `NESTED in GoG_12 (its daily means, converted by step 03)`.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=IGOG_36
$H/new_config.sh IGOG_36 4 12.2 -5.5 5.5 --res 36 --parent GoG_12 --force
grep -E "^(PARENT|HC_START|HC_END|SPINUP_DAYS|HCAST_DAYS|SPONGE|TIDES|RIVERS)=" $H/configs/$NAME/domain.cfg
bash -c "source $H/lib/common.sh $NAME >/dev/null && show_config"

## G.2 Convert the parent output (step 03 of a child)

**▶ What this cell does.** For a child, step 03 downloads nothing. It finds the finished GoG_12 cycle, converts its spin-up and hindcast daily means with `gtools/nesting.py`, writes `downloaded_data/PARENT_GoG_12/2026_05.nc` and `2026_06.nc`, and checks that the parent's ERA5 covers the period.

**Why.** This is the heart of nesting: after it, the child's input folder looks exactly like a GLORYS folder.

**✅ What to look for.** `nesting in GoG_12 (plain_tides_rivers): converting 2 daily-mean file(s)`, then `2026_05.nc: 2 record(s)` (30 and 31 May) and `2026_06.nc: 16 record(s)` (1–16 June at 12:00).

**⚠️ Classic trap.** `parent run not finished`: run the GoG_12 hindcast (Part F) first.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=IGOG_36
bash $H/steps/03_download_data.sh $NAME 2>&1 | grep -v HDF5

**▶ What this cell does.** Opens the converted June file and prints its variables, sizes and surface temperature.

**Why.** It is the proof of the idea: a file with GLORYS's names and depths, made from our own model.

**✅ What to look for.** variables `thetao, so, uo, vo, zos`; 50 depths; the GoG_12 grid size; a sensible mean SST.

In [ ]:
import os, xarray as xr
d = xr.open_dataset(os.path.expanduser("~/croco_work/hindcast/scratch/IGOG_36/downloaded_data/PARENT_GoG_12/2026_06.nc"))
print("variables:", list(d.data_vars)); print("sizes:", dict(d.sizes))
print("times:", str(d.time.values[0])[:16], "->", str(d.time.values[-1])[:16])
print("mean surface temperature %.2f degC" % float(d.thetao.isel(time=0, depth=0).mean()))

## G.3 Build the child (steps 01, 02, 04–07, 09, 10)

**▶ What this cell does.** Runs the usual steps on the child: its grid, boundaries and parameter files, ini/bry for a proof window (from the converted parent), source files, `croco.in` (with the sponge), the compile, and the child's own tide and river files. `--no-test` skips the proof run.

**Why.** Nothing is specific to nesting here, which is the point: once the parent looks like GLORYS, the child is an ordinary configuration.

**✅ What to look for.** each step `done`; `croco_plain_tides_rivers` in `scratch/IGOG_36/`; in `croco.in` the sponge line `20000. 400.` and the `online` folder of the **parent's** ERA5; `LLm0=296, MMm0=397` in `param.h` (grid 298 × 399).

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=IGOG_36
bash $H/build_config.sh $NAME --only 1
bash $H/build_config.sh $NAME --from 2 --no-test

## G.4 Run the nested hindcast ⏱

**▶ What this cell does.** Runs the child: tide file, 1-day spin-up from the parent's state on 31 May, then 15 days (1–15 June) with boundaries from the parent's daily means.

**Why.** It is the same driver as the parent's: the child's configuration tells it where the ocean comes from.

**✅ What to look for.** `[1/4] spin-up ini + bry (NEST)`, `[3/4] hindcast bry (NEST)`, `MAIN: DONE` for both runs, `HINDCAST DONE IGOG_36`.

In [ ]:
%%bash
H=~/croco_work/hindcast; NAME=IGOG_36
# nohup $H/run_hindcast_cycle.sh $NAME > ~/${NAME}_hindcast.log 2>&1 &     # <- uncomment to launch
python3 ~/croco_work/notebooks/verify_config.py $NAME plain_tides_rivers

## G.5 Does the finer grid resolve more? (the payoff)

**▶ What this cell does.** Plots the surface temperature of the parent and of the child at the same time over the child's box, and the surface relative vorticity (the spin of the currents, divided by f) of both.

**Why.** The child is only worth its cost if it resolves structures the parent cannot: sharper fronts, smaller eddies and filaments, finer coastal currents. Vorticity shows them best. Compare on day 3 or later: on the first day the child still looks like the interpolated parent.

**✅ What to look for.** similar large-scale patterns (the child is driven by the parent), with more and sharper small-scale structure in the child, above all near the coast and around the islands (Bioko, São Tomé).

In [ ]:
import os, glob, numpy as np, xarray as xr, matplotlib.pyplot as plt
def last_his(name):
    return sorted(glob.glob(os.path.expanduser(f"~/croco_work/hindcast/model-runs/{name}/20260601*/hcast/CROCO_FILES/croco_his.nc")))[-1]
def at(ds, when):
    t = np.datetime64("1993-01-01") + (ds.scrum_time.values * 1e9).astype("timedelta64[ns]")
    return int(np.argmin(np.abs(t - np.datetime64(when))))
def vort_over_f(ds, k):
    u = ds.u.isel(time=k, s_rho=-1).values; v = ds.v.isel(time=k, s_rho=-1).values
    pm, pn, f = ds.pm.values, ds.pn.values, ds.f.values
    dvdx = (v[:, 1:] - v[:, :-1]) * 0.25 * (pm[:-1, 1:] + pm[:-1, :-1] + pm[1:, 1:] + pm[1:, :-1])
    dudy = (u[1:, :] - u[:-1, :]) * 0.25 * (pn[:-1, 1:] + pn[:-1, :-1] + pn[1:, 1:] + pn[1:, :-1])
    fp = 0.25 * (f[:-1, 1:] + f[:-1, :-1] + f[1:, 1:] + f[1:, :-1])
    return (dvdx - dudy) / np.where(np.abs(fp) < 1e-6, np.nan, fp)          # f ~ 0 at the equator: masked
WHEN = "2026-06-10T00:00"
P = xr.open_dataset(last_his("GoG_12"), decode_times=False); C = xr.open_dataset(last_his("IGOG_36"), decode_times=False)
kp, kc = at(P, WHEN), at(C, WHEN)
box = [4, 12.5, -6, 5.5]
fig, ax = plt.subplots(2, 2, figsize=(13, 12), constrained_layout=True)
for j, (ds, k, name) in enumerate([(P, kp, "GoG_12 (1/12°)"), (C, kc, "IGOG_36 (1/36°)")]):
    sst = ds.temp.isel(time=k, s_rho=-1).where(ds.mask_rho == 1)
    h = ax[0, j].pcolormesh(ds.lon_rho, ds.lat_rho, sst, cmap="RdYlBu_r", vmin=24, vmax=30); ax[0, j].set_title(f"{name}: SST {WHEN[:10]}")
    z = vort_over_f(ds, k); lp = 0.25 * (ds.lon_rho.values[:-1, 1:] + ds.lon_rho.values[:-1, :-1] + ds.lon_rho.values[1:, 1:] + ds.lon_rho.values[1:, :-1])
    la = 0.25 * (ds.lat_rho.values[:-1, 1:] + ds.lat_rho.values[:-1, :-1] + ds.lat_rho.values[1:, 1:] + ds.lat_rho.values[1:, :-1])
    g = ax[1, j].pcolormesh(lp, la, z, cmap="RdBu_r", vmin=-1, vmax=1); ax[1, j].set_title(f"{name}: surface vorticity / f")
    for a in ax[:, j]: a.set_xlim(box[:2]); a.set_ylim(box[2:]); a.set_aspect("equal")
fig.colorbar(h, ax=ax[0, :], label="°C"); fig.colorbar(g, ax=ax[1, :], label="ζ / f")
plt.show()

---
# Part H — build your own region (or your own child)
1. **Describe it**: `hindcast/new_config.sh MyRegion lon_min lon_max lat_min lat_max [--res N] [--start D --end D]
   [--tides] [--rivers] [--threads N] [--hcast-days N]`. The name must start with a letter (it becomes a cpp key);
   longitudes −180..180, east positive.
2. **Grid and checks**: `build_config.sh MyRegion --only 1`, then `--only 2`, and read the step-02 output (open
   edges, Courant number, isolated water bodies).
3. **Data, build, proof**: `build_config.sh MyRegion --from 3`.
4. **Hindcast**: `run_hindcast_cycle.sh MyRegion`, then `verify_config.py MyRegion <build>`.
5. **A finer child**: `new_config.sh MyChild … --res N --parent MyRegion`, run the parent's hindcast, then
   `build_config.sh MyChild --from 2 --no-test` and `run_hindcast_cycle.sh MyChild`. The child box must lie inside
   the parent's; keep it a few parent cells away from the parent's **open** edges if you can, because the parent's
   solution right at its own boundary is mostly the boundary data.

---
## Consistency rules — guaranteed by construction (check these first if something misbehaves)
* **Name**: `# define GOG_12` (cppdefs.h) = `# elif defined GOG_12` (param.h), both from `CONFIG_NAME`.
* **Grid size**: `LLm0/MMm0` = `xi_rho − 2`, `eta_rho − 2` of `croco_grd.nc`.
* **Vertical grid**: `THETA_S THETA_B N_LEVELS SIGMA_HC` → `crocotools_param.py`, `croco.in` `S-coord`, `param.h` `N`.
* **Boundaries**: land mask → `obc.cfg` → `obc_dict`, `OBC_*`, the content of the boundary file.
* **Time**: Yorig 1993 in every ini, bry, ERA5, runoff and tide file; the first `time[DAYS]` = days since 1993-01-01.
* **Nesting**: the child box inside the parent; the parent converted with the parent's Yorig; child dt ≈ parent dt ×
  (child cell / parent cell); the child ends at least one day before the parent.

## Troubleshooting
| symptom | cause / fix |
|---|---|
| `domain.cfg not found` | create it with `hindcast/new_config.sh` |
| `open boundaries unknown` | run step 02 (or set `OBC` in domain.cfg) |
| Courant > 0.7 | set `DT` in domain.cfg, re-run from step 05 |
| GLORYS download estimated in hours | keep the default whole-period request (`CMEMS_SERVICE=arco-time-series`) |
| ERA5: `TimeoutError` or connection reset | temporary network trouble; each variable is retried 5 times, re-run to resume (finished variables are cached in `raw/`) |
| ERA5: `HTTPFileSystem requires "requests" and "aiohttp"` / `asyncio.async` syntax error | `pip install -U 'aiohttp>=3.9'` |
| `ERROR in get_bry … 'bry_time'` | the boundary file does not cover the run ± 1 day (a missing month, or a child ending too late) |
| `Abnormal termination: netCDF INPUT` after *Open Meteo file* | an ERA5 file or month is missing in `online` |
| child: `parent run not finished` / `no GoG_12 run covers …` | run the parent's hindcast first, over the child's period |
| child: `the child box must lie inside the parent` / `too close to the parent edge` | move the named edge ≥ 0.3° inside the parent (land edges too) |
| child step 04: `ERROR: The data does not cover the entire grid` | the ini/bry reader needs the parent data to cover the child grid + 0.2°: move the child edge inside, then steps 01, 02, 04 (the converted parent is kept) |
| tide file with fill values | use this package's `gtools` (fixes for domains crossing 0°) |
| `no river selected` | lower `RIVER_QMIN`, widen `RIVER_RADIUS`, or list mouths in `RIVER_EXTRA` |
| NaN / `BLOW UP` | boundaries vs mask, `DT`, `SPONGE` (always on for a child) |

---
## Verification record
_(filled in after the runs)_